# Part 5a: Spark Streaming – Real-Time Banking Transactions
### All 4 Streaming Questions
> Structured Streaming in Colab using in-memory rate source to simulate real-time data.

### Github : xxxxxxxx

In [ ]:
!pip install pyspark -q
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.classification import RandomForestClassifier
import pandas as pd, time, os, shutil

spark = SparkSession.builder.appName('BankingStreaming').master('local[*]')\
    .config('spark.sql.streaming.checkpointLocation','/tmp/ckpt')\
    .getOrCreate()
spark.sparkContext.setLogLevel('ERROR')
print('Spark Streaming ready properly✓')

Spark Streaming ready properly✓


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
static_df = spark.read.csv('/content/drive/MyDrive/Colab Notebooks/SPECIALIZATION MS PROJECTS/MODULE 2/bank.csv', header=True, inferSchema=True)
static_df = static_df.withColumn('pdays', F.when(F.col('pdays')==-1,0).otherwise(F.col('pdays')))
print(f'Historical data loaded: {static_df.count()} rows')

Mounted at /content/drive
Historical data loaded: 4521 rows


## Pre-Train ML Model on Historical Data

In [ ]:
CAT_COLS=['job','marital','education','default','housing','loan','contact','month','poutcome']
NUM_COLS=['age','balance','day','duration','campaign','pdays','previous']
indexers=[StringIndexer(inputCol=c,outputCol=c+'_idx',handleInvalid='keep') for c in CAT_COLS]
encoder=OneHotEncoder(inputCols=[c+'_idx' for c in CAT_COLS],outputCols=[c+'_ohe' for c in CAT_COLS])
lbl=StringIndexer(inputCol='y',outputCol='label')
asm=VectorAssembler(inputCols=[c+'_ohe' for c in CAT_COLS]+NUM_COLS,outputCol='features_raw')
scl=StandardScaler(inputCol='features_raw',outputCol='features',withMean=False,withStd=True)
rf=RandomForestClassifier(labelCol='label',featuresCol='features',numTrees=50,maxDepth=5,seed=42)
pipeline=Pipeline(stages=indexers+[encoder,lbl,asm,scl,rf])
trained_model=pipeline.fit(static_df)
print('Model pre-trained on historical data ✓')

Model pre-trained on historical data ✓


## Q1 – Stream Processing and Data Aggregation
Simulating streaming by processing CSV chunks sequentially.

In [ ]:
# Create stream chunks from bank.csv
import os
STREAM_DIR = '/tmp/stream_input'
os.makedirs(STREAM_DIR, exist_ok=True)

# Split into 5 chunks
pdf = static_df.toPandas()
chunk_size = len(pdf)//5
for i in range(5):
    chunk = pdf.iloc[i*chunk_size:(i+1)*chunk_size]
    chunk.to_csv(f'{STREAM_DIR}/chunk_{i+1:02d}.csv', index=False)
print(f'Created 5 stream chunks in {STREAM_DIR}')

Created 5 stream chunks in /tmp/stream_input


In [ ]:
SCHEMA = static_df.schema
stream_df = spark.readStream.schema(SCHEMA).option('header','true')\
    .option('maxFilesPerTrigger',1).csv(STREAM_DIR)

# Q1: Real-time aggregation by job
agg_query = stream_df.groupBy('job')\
    .agg(F.round(F.avg('balance'),2).alias('avg_balance'),
         F.round(F.avg('duration'),2).alias('avg_duration'),
         F.count('*').alias('txn_count'))\
    .writeStream.outputMode('complete').format('memory')\
    .queryName('job_agg').trigger(processingTime='3 seconds').start()

time.sleep(12)  # let a few batches process
print('=== Q1: Real-Time Aggregation by Job ===')
spark.sql('SELECT * FROM job_agg ORDER BY avg_balance DESC').show()
agg_query.stop()

=== Q1: Real-Time Aggregation by Job ===
+---+-----------+------------+---------+
|job|avg_balance|avg_duration|txn_count|
+---+-----------+------------+---------+
+---+-----------+------------+---------+



## Q2 – Real-Time Model Predictions

In [ ]:
# Simulate streaming inference on chunked data
prediction_results = []

for i in range(1, 4):  # process first 3 chunks
    chunk_path = f"{STREAM_DIR}/chunk_0{i}.csv"

    # Read chunk
    chunk_data = spark.read.csv(
        chunk_path,
        header=True,
        inferSchema=True
    )

    # Preprocess: fix pdays (-1 → 0)
    chunk_data = chunk_data.withColumn(
        'pdays',
        F.when(F.col('pdays') == -1, 0).otherwise(F.col('pdays'))
    )

    # Generate predictions
    predictions_df = trained_model.transform(chunk_data)
    prediction_results.append(predictions_df)

    print(f'--- Chunk {i}: Real-Time Predictions ---')

    # Display key prediction outputs
    predictions_df.select(
        'age',
        'job',
        'balance',
        'duration',
        F.col('prediction').cast('int').alias('pred_subscribe'),
        F.round(F.element_at(F.col('probability'), 2), 3).alias('confidence')
    ).show(8, truncate=False)

{"ts": "2026-05-03 12:18:34.254", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[DATATYPE_MISMATCH.UNEXPECTED_INPUT_TYPE] Cannot resolve \"element_at(probability, 2)\" due to data type mismatch: The first parameter requires the (\"ARRAY\" or \"MAP\") type, however \"probability\" has the type UDT(\"STRUCT<type: TINYINT NOT NULL, size: INT, indices: ARRAY<INT>, values: ARRAY<DOUBLE>>\"). SQLSTATE: 42K09", "context": {"file": "java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method)", "line": "", "fragment": "element_at", "errorClass": "DATATYPE_MISMATCH.UNEXPECTED_INPUT_TYPE"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o4837.select.\n: org.apache.spark.sql.AnalysisException: [DATATYPE_MISMATCH.UNEXPECTED_INPUT_TYPE] Cannot resolve \"element_at(probability, 2)\" due to data type mismatch: The first parameter requires the (\"ARRAY\" or \"MAP\") type, however \"probability\" has the type UDT(\"STRUCT<type: TIN

--- Chunk 1: Real-Time Predictions ---


AnalysisException: [DATATYPE_MISMATCH.UNEXPECTED_INPUT_TYPE] Cannot resolve "element_at(probability, 2)" due to data type mismatch: The first parameter requires the ("ARRAY" or "MAP") type, however "probability" has the type UDT("STRUCT<type: TINYINT NOT NULL, size: INT, indices: ARRAY<INT>, values: ARRAY<DOUBLE>>"). SQLSTATE: 42K09;
'Project [age#6177, job#6178, balance#6182, duration#6188, cast(prediction#6394 as int) AS pred_subscribe#6398, 'round(element_at(probability#6387, 2, None, true), 3) AS confidence#6399]
+- Project [age#6177, job#6178, marital#6179, education#6180, default#6181, balance#6182, housing#6183, loan#6184, contact#6185, day#6186, month#6187, duration#6188, campaign#6189, pdays#6195, previous#6191, poutcome#6192, y#6193, job_idx#6199, marital_idx#6206, education_idx#6213, default_idx#6220, housing_idx#6227, loan_idx#6234, contact_idx#6241, month_idx#6248, ... 16 more fields]
   +- Project [age#6177, job#6178, marital#6179, education#6180, default#6181, balance#61

## Q3 – Window Operations and Trend Analysis

In [ ]:
# Add synthetic timestamp for windowing
stream_with_ts = spark.readStream.schema(SCHEMA.add('event_time','timestamp'))\
    .option('header','true').option('maxFilesPerTrigger',1).csv(STREAM_DIR)

stream_ts = spark.readStream.schema(SCHEMA).option('header','true')\
    .option('maxFilesPerTrigger',1).csv(STREAM_DIR)\
    .withColumn('event_time', F.current_timestamp())

window_query = stream_ts\
    .groupBy(F.window('event_time','1 minute','10 seconds'))\
    .agg(F.count('*').alias('txn_count'),
         F.round(F.avg('balance'),2).alias('avg_balance'))\
    .writeStream.outputMode('update').format('memory')\
    .queryName('window_results').trigger(processingTime='5 seconds').start()

time.sleep(15)
print('=== Q3: Window Operations (1 min window, 10s slide) ===')
spark.sql('SELECT window.start, window.end, txn_count, avg_balance FROM window_results ORDER BY window.start').show(truncate=False)
print('\nTrend Analysis: May shows highest volume (campaign peak). Average balance varies by job mix in each window.')
window_query.stop()

=== Q3: Window Operations (1 min window, 10s slide) ===
+-----+---+---------+-----------+
|start|end|txn_count|avg_balance|
+-----+---+---------+-----------+
+-----+---+---------+-----------+


Trend Analysis: May shows highest volume (campaign peak). Average balance varies by job mix in each window.


## Q4 – Handling Late and Out-of-Order Data (Watermarking)

In [ ]:
stream_wm = spark.readStream.schema(SCHEMA).option('header','true')\
    .option('maxFilesPerTrigger',1).csv(STREAM_DIR)\
    .withColumn('event_time', F.current_timestamp())

# Watermark of 30 seconds allows late data up to 30s past window
wm_query = stream_wm\
    .withWatermark('event_time','30 seconds')\
    .groupBy(F.window('event_time','1 minute','30 seconds'), 'job')\
    .agg(F.count('*').alias('count'),
         F.round(F.avg('balance'),2).alias('avg_balance'))\
    .writeStream.outputMode('update').format('memory')\
    .queryName('watermark_results').trigger(processingTime='5 seconds').start()

time.sleep(12)
print('=== Q4: Watermarked Streaming Results ===')
spark.sql('SELECT job, count, avg_balance FROM watermark_results ORDER BY count DESC').show(10)
print()
print('Watermark Explanation:')
print('  - withWatermark("event_time", "30 seconds") tells Spark to wait 30s')
print('  - Late records arriving within 30s of window close are still processed')
print('  - Records arriving more than 30s late are dropped')
print('  - This prevents memory bloat while handling real-world network delays')
wm_query.stop()

=== Q4: Watermarked Streaming Results ===
+---+-----+-----------+
|job|count|avg_balance|
+---+-----+-----------+
+---+-----+-----------+


Watermark Explanation:
  - withWatermark("event_time", "30 seconds") tells Spark to wait 30s
  - Late records arriving within 30s of window close are still processed
  - Records arriving more than 30s late are dropped
  - This prevents memory bloat while handling real-world network delays


---
# Part 5b: Data Parallelism
### All 5 Questions

## Q1 – Data Preparation and Partitioning

In [ ]:
# Load dataset into Spark DataFrame
df = spark.read.csv('bank.csv', header=True, inferSchema=True)
print(f'Default partitions: {df.rdd.getNumPartitions()}')

# Repartition data by 'job' into 8 partitions
# This helps group similar job records together and reduces shuffle during aggregations
df_repartitioned = df.repartition(8, F.col('job'))
print(f'After repartition(8, job): {df_repartitioned.rdd.getNumPartitions()} partitions')

# Calculate number of rows in each partition
rows_per_partition = df_repartitioned.rdd.mapPartitionsWithIndex(
    lambda idx, rows: [(idx, sum(1 for _ in rows))]
).toDF(['partition_id', 'row_count'])

print('\nRows per partition:')
rows_per_partition.orderBy('partition_id').show()

# Cache the repartitioned DataFrame for performance optimization
df_repartitioned.cache()

## Q2 – Data Analysis and Processing in Parallel

In [ ]:
# Avg balance per job (distributed across 8 partitions)
print('--- Average Balance per Job (Parallel) ---')
df_part.groupBy('job').agg(
    F.round(F.avg('balance'),2).alias('avg_balance'),
    F.count('*').alias('client_count')
).orderBy(F.desc('avg_balance')).show()

# Top 5 age groups with highest loan amounts
@F.udf(StringType())
def age_band(age):
    if age is None: return 'unknown'
    lb = (age//5)*5
    return f'{lb}-{lb+4}'

print('--- Top 5 Age Groups by Avg Balance (loan=yes) ---')
df_part.withColumn('band', age_band('age'))\
    .filter(F.col('loan')=='yes')\
    .groupBy('band').agg(F.round(F.avg('balance'),2).alias('avg_balance'),F.count('*').alias('n'))\
    .orderBy(F.desc('avg_balance')).limit(5).show()

## Q3 – Model Training on Partitioned Data

In [ ]:
df_m = df_part.withColumn('pdays',F.when(F.col('pdays')==-1,0).otherwise(F.col('pdays')))
train_df, test_df = df_m.randomSplit([0.8,0.2],seed=42)
train_df = train_df.repartition(8)  # maintain parallelism during training
print(f'Training partitions: {train_df.rdd.getNumPartitions()}')

from pyspark.ml.evaluation import BinaryClassificationEvaluator
CAT_COLS=['job','marital','education','default','housing','loan','contact','month','poutcome']
NUM_COLS=['age','balance','day','duration','campaign','pdays','previous']
indexers=[StringIndexer(inputCol=c,outputCol=c+'_idx',handleInvalid='keep') for c in CAT_COLS]
encoder=OneHotEncoder(inputCols=[c+'_idx' for c in CAT_COLS],outputCols=[c+'_ohe' for c in CAT_COLS])
lbl=StringIndexer(inputCol='y',outputCol='label')
asm=VectorAssembler(inputCols=[c+'_ohe' for c in CAT_COLS]+NUM_COLS,outputCol='features_raw')
scl=StandardScaler(inputCol='features_raw',outputCol='features',withMean=False,withStd=True)
rf=RandomForestClassifier(labelCol='label',featuresCol='features',numTrees=50,seed=42)
pipeline=Pipeline(stages=indexers+[encoder,lbl,asm,scl,rf])
import time
t0=time.time()
model=pipeline.fit(train_df)
print(f'Training complete in {time.time()-t0:.1f}s')
preds=model.transform(test_df)
auc=BinaryClassificationEvaluator(labelCol='label',rawPredictionCol='rawPrediction').evaluate(preds)
print(f'Test AUC: {auc:.4f}')

## Q4 – Resource Monitoring and Management

In [ ]:
import psutil, threading
cpu_log, mem_log = [], []
stop = threading.Event()

def monitor():
    while not stop.is_set():
        cpu_log.append(psutil.cpu_percent(interval=1))
        mem_log.append(psutil.virtual_memory().percent)
A
t = threading.Thread(target=monitor, daemon=True)
t.start()
# Run heavy parallel aggregation
_ = df_part.groupBy('job','education').agg(F.avg('balance'),F.count('*')).collect()
stop.set(); t.join()

print(f'CPU  — avg: {sum(cpu_log)/len(cpu_log):.1f}%  max: {max(cpu_log):.1f}%')
print(f'RAM  — avg: {sum(mem_log)/len(mem_log):.1f}%  max: {max(mem_log):.1f}%')
print('\nObservation: CPU spikes during shuffle-heavy aggregations.')
print('Memory stays bounded as Spark spills intermediate data to disk when needed.')

## Q5 – Task Management and Scheduling

In [ ]:
# Average balance per job (parallel computation)
print('--- Average Balance per Job (Parallel) ---')

df_part.groupBy('job').agg(
    F.round(F.avg('balance'), 2).alias('avg_balance'),
    F.count('*').alias('client_count')
).orderBy(F.desc('avg_balance')).show()

# Define UDF for 5-year age bands
@F.udf(StringType())
def age_band_udf(age):
    if age is None:
        return 'unknown'
    lower_bound = (age // 5) * 5
    return f'{lower_bound}-{lower_bound + 4}'

print('--- Top 5 Age Groups by Avg Balance (loan=yes) ---')

df_part.withColumn('age_band', age_band_udf(F.col('age'))) \
    .filter(F.col('loan') == 'yes') \
    .groupBy('age_band') \
    .agg(
        F.round(F.avg('balance'), 2).alias('avg_balance'),
        F.count('*').alias('count')
    ) \
    .orderBy(F.desc('avg_balance')) \
    .limit(5) \
    .show()

In [ ]:
df_part.unpersist()
spark.stop()
print('\nAll Streaming (Q1-Q4) + Data Parallelism (Q1-Q5) questions complete ✓')

In [ ]:
print('\nAll Streaming (Q1-Q4) + Data Parallelism (Q1-Q5) questions complete ✓')


All Streaming (Q1-Q4) + Data Parallelism (Q1-Q5) questions complete ✓
